In [6]:
import os 
import sys
import json
import random
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

load_dotenv(project_root / ".env")


openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

from real_state.config import (
    CRAWL_OUT_DIR, VECTOR_DIR, EMBEDDING_MODEL, PROVIDER
)

random.seed(42)

provider = "openrouter" if openrouter_key else "OpenAI"
print("✅ Environment loaded")
print(f"🌐 Provider: {provider}")
print(f"📁 Project root: {project_root}")

✅ Environment loaded
🌐 Provider: openrouter
📁 Project root: d:\ai\bootcamp\mini-project-02\real_state


## Import chunking Services


In [7]:
from real_state.application.ingest_document_service import (
    sementic_chunk,
    fixed_chunk,
    sliding_chunks,
    parent_child_chunk,
    late_chunk_index,
    late_chunk_split
)

## Load Corpus

In [8]:
jsonl_path = CRAWL_OUT_DIR / "primelands_corpus.jsonl"

if not jsonl_path.exists():
    raise FileExistsError(f"❌ Corpus not found.")

with open(jsonl_path, 'r', encoding='utf-8') as f:
    documents = [json.loads(line) for line in f]

print(f"✅ Loaded {len(documents)} documents")
print(f"📊 Total content size: {sum(len(d['content']) for d in documents):,} chars")

✅ Loaded 207 documents
📊 Total content size: 381,130 chars


## Apply chunking strategies

In [9]:
import shutil

if VECTOR_DIR.exists():
    print(f"🗑️  Removing existing vector store: {VECTOR_DIR}")
    shutil.rmtree(VECTOR_DIR)
    print("   ✅ Cleaned up")

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Fresh vector directory ready: {VECTOR_DIR}")


🗑️  Removing existing vector store: d:\ai\bootcamp\mini-project-02\real_state\data\vector_store
   ✅ Cleaned up
📁 Fresh vector directory ready: d:\ai\bootcamp\mini-project-02\real_state\data\vector_store


In [10]:
print("🔄 Running semantic chunking...")
semantic_chunks = sementic_chunk(documents)


semantic_path = CRAWL_OUT_DIR / "chunks_semantic.jsonl"
with open(semantic_path, 'w', encoding='utf-8') as f:
    for chunk in semantic_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Semantic chunking complete: {len(semantic_chunks)} chunks")
print(f"💾 Saved to: {semantic_path}")

🔄 Running semantic chunking...
✅ Semantic chunking complete: 207 chunks
💾 Saved to: d:\ai\bootcamp\mini-project-02\real_state\data\chunks_semantic.jsonl


In [11]:
print("🔄 Running fixed-window chunking...")
fixed_chunks = fixed_chunk(documents)

fixed_path = CRAWL_OUT_DIR / "chunks_fixed.jsonl"
with open(fixed_path, 'w', encoding='utf-8') as f:
    for chunk in fixed_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

avg_tokens = sum(c['token_count'] for c in fixed_chunks) / len(fixed_chunks) if fixed_chunks else 0
print(f"✅ Fixed chunking complete: {len(fixed_chunks)} chunks")
print(f"📊 Avg token count: {avg_tokens:.1f}")
print(f"💾 Saved to: {fixed_path}")

🔄 Running fixed-window chunking...
✅ Fixed chunking complete: 210 chunks
📊 Avg token count: 462.1
💾 Saved to: d:\ai\bootcamp\mini-project-02\real_state\data\chunks_fixed.jsonl


In [12]:
print("🔄 Running sliding-window chunking...")
sliding_chunks = sliding_chunks(documents)

sliding_path = CRAWL_OUT_DIR / "chunks_sliding.jsonl"
with open(sliding_path, 'w', encoding="utf-8") as f:
    for chunk in sliding_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Sliding chunking complete: {len(sliding_chunks)} chunks")
print(f"💾 Saved to: {sliding_path}")

🔄 Running sliding-window chunking...
✅ Sliding chunking complete: 482 chunks
💾 Saved to: d:\ai\bootcamp\mini-project-02\real_state\data\chunks_sliding.jsonl


In [13]:
print("🔄 Running parent-child chunking...")
child_chunks, parent_chunks = parent_child_chunk(documents)


parent_path = CRAWL_OUT_DIR / "parent_chunk.jsonl"
child_path = CRAWL_OUT_DIR / "child_chunk.jsonl"

with open(parent_path, "w", encoding="utf-8") as f:
    for chunk in parent_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

with open(child_path, "w", encoding="utf-8") as f:
    for chunk in child_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Parent child chunking complete: \nparent len : {len(parent_chunks)}\nchild len: {len(child_chunks)}")
print(f"💾 save to: \nparent path: {parent_path}\nchild chunks: {child_path}")
        

🔄 Running parent-child chunking...
✅ Parent child chunking complete: 
parent len : 207
child len: 537
💾 save to: 
parent path: d:\ai\bootcamp\mini-project-02\real_state\data\parent_chunk.jsonl
child chunks: d:\ai\bootcamp\mini-project-02\real_state\data\child_chunk.jsonl


## Spot-Checks Sample

In [14]:
print("Spot-check: 2 samples from each strategy\n")

def print_sample(chunk, strategy_name):
    print(f"**{strategy_name}** chunk:")
    print(f" URL: {chunk['url']}")
    print(f" Strategy: {chunk['strategy']}")
    print(f" Text length: {len(chunk['text'])} chars")
    print(f" Preview: {chunk['text'][500:]}")
    print()

print("=" * 60)
print("SEMENTIC SAMPLES")
print("=" * 60)

for chunk in random.sample(semantic_chunks, min(2, len(semantic_chunks))):
    print_sample(chunk, "Semantic")


print("=" * 60)
print("FIXED SAMPLES")
print("=" * 60)

for chunk in random.sample(fixed_chunks, min(2, len(fixed_chunks))):
    print_sample(chunk, "Fixed")
    

print("=" * 60)
print("SLIDING SAMPLES")
print("=" * 60)

for chunk in random.sample(sliding_chunks, min(2, len(sliding_chunks))):
    print_sample(chunk, "Sliding")




Spot-check: 2 samples from each strategy

SEMENTIC SAMPLES
**Semantic** chunk:
 URL: https://www.primelands.lk/land/PRIME-CAPITAL-HORANA/en
 Strategy: semantic
 Text length: 1670 chars
 Preview: in road
* Close to industrial zone

## Payment Plan

Pay a 10% Down Payment & the Remaining Balance can be paid within 40 months

Pay a 15% Down Payment & the Remaining Balance can be paid within 12 months without interest.

Pay a 25% Down Payment & the Remaining Balance can be paid within 18 months without interest.

Pay a 20% Down Payment & the Remaining Balance can be paid within 12 months without interest.

Bank loan facilities can be arranged from Prime Group.

![brochure_black.png](https://www.primelands.lk/public/assets/images/brochure_black.png)
![brochure_white.png](https://www.primelands.lk/public/assets/images/brochure_white.png)

Brochure

## Facilities

![210603110622elec.jpg](https://plcms.primelands.lk/images/210603110622elec.jpg)

ELECTRICITY

![210603110640water.jpg](https://pl

In [15]:
from real_state.infrastructure.llm_provider.embeddings import get_default_embeddings

In [16]:
from real_state.infrastructure.db.vector_db import QdrantStorage
from real_state.infrastructure.llm_provider.embeddings import get_default_embeddings

emb = get_default_embeddings()

db_client = QdrantStorage(embedding=emb) 

all_chunks = semantic_chunks + fixed_chunks + sliding_chunks




In [17]:
all_chunks = semantic_chunks + fixed_chunks + sliding_chunks
# db_client.upsert_chunks(all_chunks)

## Index Sanity Check

In [18]:
print(" Index sanity check\n")
info = db_client.collection_info()
print(info)

 Index sanity check

{'name': 'prime_lands', 'points_count': 899, 'index_vectors_count': 0, 'vector_size': 3072, 'distance': 'COSINE', 'status': 'GREEN'}


In [19]:
query = "how much SKYE BLOSSOM - KOTTAWA?"
db_client.search_chunks(query=query)

[{'text': "Project ID: 75\n\n# SKYE BLOSSOM - KOTTAWA\n\n# 50,000,000 LKR\n\n## About the Project\n\nDiscover Skye Blossom Kottawa – The Home That Grows with You.\n\nNestled in the heart of Kottawa, Skye Blossom redefines refined living and investment excellence. This prestigious address enjoys exceptionally high rental demand, driven by the area's dynamic growth and urban sophistication.\n\nA rare investment opportunity is rising in Kottawa, the only high-rise of its kind in this thriving suburb. Skye Blossom Kottawa is poised for strong value appreciation in one of the fastest-growing urban hubs, making it ideal for investors expanding their portfolio or homeowners seeking lasting value.\n\nApartment living often means sacrificing greenery and open spaces, but not at Skye Blossom Kottawa. With lush, greenery-filled balconies, these twin 16-story towers create a vertical oasis, blending modern design with nature. More than just a home, it’s a thriving asset that grows in value.\n\nPre